# Model Selection and Regularization: California Housing Dataset

In this lab, we compare statistical models to determine the best approach for predicting Median House Value.

We will cover:
- **AIC & BIC** for model selection (comparing models with different predictor sets)
- **Ridge & Lasso Regression** for regularization (reducing overfitting)
- **MSE** as our evaluation metric for predictive accuracy

In [1]:
# CodeGrade step0

from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import os
import tarfile
import joblib
import statsmodels.api as sm
from sklearn.datasets._base import _pkl_filepath, get_data_home

# Process to get the data to be as it is in scikit-learn
archive_path = "cal_housing.tgz"
data_home = get_data_home(data_home=None)
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, 'cal_housing.pkz')

with tarfile.open(mode="r:gz", name=archive_path) as f:
    cal_housing = np.loadtxt(
        f.extractfile('CaliforniaHousing/cal_housing.data'),
        delimiter=',')
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]

    joblib.dump(cal_housing, filepath, compress=6)

# Load the dataset
california = fetch_california_housing(as_frame=True)
data = california.data
target = california.target

In [2]:
# CodeGrade step0.1
X = data[['MedInc', 'AveRooms', 'AveOccup']]
y = target

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## Step 1: Full Model — AIC and BIC

We add a constant to `X` (so statsmodels fits an intercept), fit an OLS model 
using all available features, and extract the **AIC** and **BIC** as model 
selection criteria.

- **AIC** (Akaike Information Criterion): penalizes model complexity, 
  rewards goodness of fit. Lower is better.
- **BIC** (Bayesian Information Criterion): applies a heavier penalty for 
  additional parameters than AIC. Lower is better.

In [ ]:
# CodeGrade step1

X_const = sm.add_constant(X)
full_model = sm.OLS(y, X_const).fit()
round(full_model.aic), round(full_model.bic)

(50962, 50994)

## Step 2: Subset Model Comparison — AIC and BIC

Instead of using all features, we fit two smaller OLS models using only 
selected predictors and compare their AIC and BIC:

- **subset1**: `MedInc` and `AveRooms`
- **subset2**: `MedInc` and `AveOccup`

This tells us which combination of features gives us a better-fitting, 
more parsimonious model.

In [ ]:
# CodeGrade step2

# Subset 1
X_subset1 = sm.add_constant(X[['MedInc', 'AveRooms']])
subset1_model = sm.OLS(y, X_subset1).fit()

# Subset 2
X_subset2 = sm.add_constant(X[['MedInc', 'AveOccup']])
subset2_model = sm.OLS(y, X_subset2).fit()

round(subset1_model.aic), round(subset1_model.bic), round(subset2_model.aic), round(subset2_model.bic)

(51016, 51040, 51199, 51222)

## Step 3: Ridge Regression — Prediction and MSE

Ridge regression adds an **L2 penalty** (sum of squared coefficients) to the 
loss function, shrinking all coefficients toward zero to reduce overfitting 
without eliminating any feature entirely.

We fit Ridge on the training set, predict on `X_test`, and compute the 
**Mean Squared Error (MSE)** to evaluate predictive performance.

In [5]:
# CodeGrade step3

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_pred)
round(ridge_mse, 4)

0.7007

## Step 4: Ridge Coefficients

Inspecting the coefficients of the Ridge model shows how much each feature 
contributes to the predicted house value after regularization. 
Because Ridge uses L2 shrinkage, all coefficients remain in the model 
but are pulled toward zero.

In [6]:
# CodeGrade step4
ridge.coef_

array([ 0.43687436, -0.04042693, -0.00382598])

## Step 5: Lasso Regression — Prediction and MSE

Lasso regression adds an **L1 penalty** (sum of absolute coefficients) to the 
loss function. Unlike Ridge, Lasso can shrink some coefficients all the way to 
**zero**, effectively performing automatic feature selection.

We fit Lasso on the training set, predict on `X_test`, and compute the MSE.

In [ ]:
# CodeGrade step5

from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)
lasso.fit(X_train, y_train)
lasso_pred = lasso.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_pred)
round(lasso_mse, 4)

0.7047